# 00 - Effective surface area versus the E(B-V) cut, with the official rubin_sim MAF

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-29
- Kernel: conda_py313_opsim53
- Context: series `10_DESCMAFDEPTHANDGALCOUNT`. **No new metric is written in this notebook.** It reuses the effective-area computation of the official batch `rubin_sim/maf/batches/science_radar_batch.py` (Cosmology / Static Science) and applies it to the 7 simulations of the series (`shrink_fp_dust_{0.050, 0.080, 0.120, 0.150, 0.199, 0.250}_v5.3.6_10yrs.db` and `baseline_v5.3.6_10yrs.db`, E(B-V) threshold 0.200).
- Previous version of this notebook (custom `EffectiveSurfaceAreaMetric` weighted by the seeing-aware n_eff model): `00_EffectiveSurfaceArea_custom_neff_old.ipynb`.

## Notebook overview

**What the official batch does.** In the block *Cosmology / Static Science* of `science_radar_batch`, the effective survey area is not a dedicated metric: it is a *summary statistic* attached to the depth metric `ExgalM5WithCuts`:

```python
m = metrics.ExgalM5WithCuts(lsst_filter='i', n_filters=6, extinction_cut=lim_ebv, depth_cut=ptsrc_lim_mag_i_band)
summaryMetrics = [..., metrics.CountRatioMetric(norm_val=1 / pix_area, metric_name='Effective Area (deg)')]
bundle = mb.MetricBundle(m, slicer, sqlconstraint, maps_list=[dustmap], summary_metrics=summaryMetrics, ...)
```

- `ExgalM5WithCuts` returns the dust-corrected coadded depth of the `i` band in the pixels that pass the cuts (E(B-V) below `extinction_cut`, coverage in `n_filters` = 6 bands, coadded `i` depth above `depth_cut`), and the bad value elsewhere. Section 3 prints its source so the cuts can be checked in the installed version.
- `CountRatioMetric(norm_val=1/pix_area)` counts the valid (unmasked) pixels and divides by `1/pix_area`, i.e. **A_eff = N_valid_pixels x pixel_area**. The batch label says `(deg)` but the unit is deg^2.

**Settings taken from the batch (year 10).** `lsst_filter='i'`, `n_filters=6`, `depth_cut = 26.0 - 0.1 = 25.9`, non-DDF visits, `night <= 10 x 365.25 + 0.5`, `nside=64`, `DustMap(interp=False)`.

**What is varied.** The batch uses `lim_ebv = 0.2` for every run. Here `extinction_cut` is set to the E(B-V) threshold of each run (`USE_RUN_CUT = True`), as in the other notebooks of the series. Set `USE_RUN_CUT = False` to keep the batch value 0.2 for all runs.

**Outputs**: cached maps and tables in `data_00_EFFAREA/`, figures in `figs_00_EFFAREA/`, in particular the plot of A_eff normalized to the baseline (10 years).

## 1. Imports

In [ ]:
import os
import inspect
from os.path import join, isfile

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.maps as maf_maps
import rubin_sim.maf.metrics as metrics
import rubin_sim.maf.metric_bundles as mb

print("rubin_sim version:", rubin_sim.__version__)

## 2. Configuration

In [ ]:
NB_TAG = "EFFAREA"
data_dir = f"data_00_{NB_TAG}"
figs_dir = f"figs_00_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
resultsDb = maf.db.ResultsDb(out_dir=data_dir)

OPSIM_DIR = "/Users/dagoret/DATA/OpSim"
RUNS_INFO = [
    ("shrink_fp_dust_0.050_v5.3.6_10yrs", 0.050),
    ("shrink_fp_dust_0.080_v5.3.6_10yrs", 0.080),
    ("shrink_fp_dust_0.120_v5.3.6_10yrs", 0.120),
    ("shrink_fp_dust_0.150_v5.3.6_10yrs", 0.150),
    ("shrink_fp_dust_0.199_v5.3.6_10yrs", 0.199),
    ("baseline_v5.3.6_10yrs", 0.200),
    ("shrink_fp_dust_0.250_v5.3.6_10yrs", 0.250),
]
RUN_NAMES = [r for r, _ in RUNS_INFO]
EBV_RUN = dict(RUNS_INFO)  # E(B-V) threshold that defines the footprint of each run (x axis of the plots)
REF_RUN = "baseline_v5.3.6_10yrs"


def get_db_path(run_name):
    fname = run_name + ".db"
    candidates = [join(OPSIM_DIR, fname), join(OPSIM_DIR, "sim_baseline", fname)]
    for c in candidates:
        if isfile(c):
            return c
    raise FileNotFoundError(f"OpSim db not found for run {run_name}. Tried: {candidates}")


# --- parameters of the Static Science block of science_radar_batch, year 10
NSIDE = 64
BAND = "i"
N_FILTERS = 6  # coverage required in ugrizy
DEPTH_CUT = 26.0 - 0.1  # mag_cuts[10] of the batch: minimum coadded i depth
BATCH_EBV = 0.2  # lim_ebv of the batch
USE_RUN_CUT = False  # True: extinction_cut = threshold of each run; False: 0.2 for all runs
EXTINCTION_CUT = {r: (c if USE_RUN_CUT else BATCH_EBV) for r, c in RUNS_INFO}
MAX_NIGHT = 10 * 365.25 + 0.5
SQL = "night <= %s and scheduler_note not like %s" % (
    MAX_NIGHT,
    repr("DD%"),
)  # all bands are kept (n_filters=6)
EFFAREA_NAME = "Effective Area (deg)"  # name of the summary metric in the batch (unit: deg^2)
FORCE_RECOMPUTE = False

pix_area = hp.nside2pixarea(NSIDE, degrees=True)  # deg^2
print(f"nside={NSIDE}, pixel area = {pix_area:.5f} deg^2, depth_cut = {DEPTH_CUT:g}, n_filters = {N_FILTERS}")
print("extinction_cut per run:", EXTINCTION_CUT)
print("sql:", SQL)

## 3. The official metrics used: signatures and sources

Printed from the installed `rubin_sim`, to check exactly which cuts `ExgalM5WithCuts` applies and how `CountRatioMetric` normalizes.

In [ ]:
print(inspect.signature(metrics.ExgalM5WithCuts))
print(inspect.getsource(metrics.ExgalM5WithCuts))
print("-" * 80)
print(inspect.getsource(metrics.CountRatioMetric))

## 4. Run the official MAF on the 7 simulations

One `MetricBundle` per run, built exactly as in the batch. The map of `ExgalM5WithCuts` and the value of the summary statistic `Effective Area (deg)` are cached.

In [ ]:
dustmap = maf_maps.DustMap(nside=NSIDE, interp=False)


def bundle_to_masked(bundle):
    data = np.ma.getdata(bundle.metric_values).astype(float)
    mask = np.ma.getmaskarray(bundle.metric_values) | ~np.isfinite(data) | (data < -600.0)
    return np.ma.masked_array(data, mask=mask)


def cache_path(run_name):
    tag = run_name.replace(".", "_")
    return join(
        data_dir,
        f"EffArea_{tag}_{BAND}_nf{N_FILTERS}_m5cut{DEPTH_CUT:g}_ebv{EXTINCTION_CUT[run_name]:g}_nside{NSIDE}.npz",
    )


def run_official(run_name):
    path = cache_path(run_name)
    if isfile(path) and not FORCE_RECOMPUTE:
        print(f"[{run_name}] loaded cached map")
        with np.load(path) as f:
            return np.ma.masked_array(f["data"], mask=f["mask"]), float(f["a_eff_summary"])
    dbpath = get_db_path(run_name)
    print(f"[{run_name}] running ExgalM5WithCuts on {dbpath}")
    metric = metrics.ExgalM5WithCuts(
        lsst_filter=BAND,
        n_filters=N_FILTERS,
        extinction_cut=EXTINCTION_CUT[run_name],
        depth_cut=DEPTH_CUT,
    )
    summary = [metrics.CountRatioMetric(norm_val=1.0 / pix_area, metric_name=EFFAREA_NAME)]
    slicer = slicers.HealpixSlicer(nside=NSIDE, use_cache=False)
    bundle = mb.MetricBundle(
        metric,
        slicer,
        SQL,
        maps_list=[dustmap],
        run_name=run_name,
        info_label=f"{BAND} band non-DD 10yr",
        summary_metrics=summary,
    )
    group = mb.MetricBundleGroup(
        mb.make_bundles_dict_from_list([bundle]), dbpath, out_dir=data_dir, results_db=resultsDb
    )
    group.run_all()
    if not bundle.summary_values and hasattr(group, "summary_all"):
        group.summary_all()
    m = bundle_to_masked(bundle)
    sv = bundle.summary_values or {}
    a_sum = next((float(v) for k, v in sv.items() if "Effective Area" in k), np.nan)
    np.savez_compressed(path, data=np.ma.getdata(m), mask=np.ma.getmaskarray(m), a_eff_summary=a_sum)
    return m, a_sum


results = {run_name: run_official(run_name) for run_name in RUN_NAMES}
maps_m5 = {r: v[0] for r, v in results.items()}  # coadded dust-corrected i depth on the valid pixels
a_eff_maf = {r: v[1] for r, v in results.items()}  # value of the summary metric 'Effective Area (deg)'

## 5. Effective area per run

`A_eff (MAF summary)` is the value returned by the official summary metric. `A_eff (pixels x area)` recomputes it from the map (number of valid pixels times the pixel area); the two must agree.

In [ ]:
rows = []
for run_name in RUN_NAMES:
    m = maps_m5[run_name]
    n = int(m.count())
    rows.append(
        {
            "run": run_name,
            "E(B-V) threshold": EBV_RUN[run_name],
            "extinction_cut used": EXTINCTION_CUT[run_name],
            "n_pixels": n,
            "A_eff (pixels x area) [deg2]": n * pix_area,
            "A_eff (MAF summary) [deg2]": a_eff_maf[run_name],
            "mean coadd m5 (i)": float(m.mean()) if n else np.nan,
        }
    )
df = pd.DataFrame(rows).set_index("run")
df["A_eff [deg2]"] = df["A_eff (MAF summary) [deg2]"].fillna(df["A_eff (pixels x area) [deg2]"])
df["A_eff / baseline"] = df["A_eff [deg2]"] / df.loc[REF_RUN, "A_eff [deg2]"]
df["dA_eff vs previous threshold [deg2]"] = df["A_eff [deg2]"].diff()
df.to_csv(join(data_dir, "EffectiveArea_official_per_run_summary.csv"))

ok = df["A_eff (MAF summary) [deg2]"].notna()
if ok.any():
    diff = (df.loc[ok, "A_eff (MAF summary) [deg2]"] - df.loc[ok, "A_eff (pixels x area) [deg2]"]).abs().max()
    print(f"max |summary - pixels x area| = {diff:.3g} deg2")
else:
    print("WARNING: the summary metric value was not found in bundle.summary_values; using pixels x area")
display(df.round(4))

## 6. Normalization to the baseline (10 years)

Left: effective area versus the E(B-V) threshold that defines the footprint. Right: the same quantity divided by the effective area of the baseline run (10 years, E(B-V) = 0.200).

In [ ]:
def save_fig(fig, name):
    base = join(figs_dir, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")


x = np.array([EBV_RUN[r] for r in RUN_NAMES])
a = df["A_eff [deg2]"].values
a_norm = df["A_eff / baseline"].values
i_ref = RUN_NAMES.index(REF_RUN)

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
axs[0].plot(x, a, "o-", color="tab:blue", label="A_eff")
axs[0].plot(x[i_ref], a[i_ref], "o", mfc="none", mec="k", ms=13, label="baseline (10 yr)")
axs[0].set_ylabel("Effective area A_eff [deg$^2$]")
axs[0].set_title("ExgalM5WithCuts + CountRatioMetric (official MAF)")

axs[1].plot(x, a_norm, "o-", color="tab:blue", label="A_eff / A_eff(baseline)")
axs[1].axhline(1.0, color="gray", ls="--", lw=0.8)
axs[1].plot(x[i_ref], 1.0, "o", mfc="none", mec="k", ms=13, label="baseline (10 yr)")
for xi, yi in zip(x, a_norm):
    axs[1].annotate(f"{yi:.3f}", (xi, yi), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
axs[1].set_ylabel("A_eff / A_eff(baseline, 10 yr)")
axs[1].set_title("Normalized to the baseline (10 yr)")

for ax in axs:
    ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle(
    f"Effective area: {BAND} band, n_filters={N_FILTERS}, depth cut {DEPTH_CUT:g}, extinction_cut = "
    + ("threshold of each run" if USE_RUN_CUT else f"{BATCH_EBV:g}"),
    fontsize=11,
)
fig.tight_layout()
save_fig(fig, "EFFAREA_vs_dust_threshold_normalized")
plt.show()

## 7. Footprint maps

Coadded dust-corrected `i` depth returned by `ExgalM5WithCuts` on the pixels that define the effective area (common color scale). The number of colored pixels times the pixel area is A_eff.

In [ ]:
pooled = np.concatenate([m.compressed() for m in maps_m5.values() if m.count()])
vmin, vmax = np.percentile(pooled, 1), np.percentile(pooled, 99)
fig = plt.figure(figsize=(15, 12))
for i, run_name in enumerate(RUN_NAMES, start=1):
    hp.mollview(
        np.ma.filled(maps_m5[run_name], hp.UNSEEN),
        fig=fig.number,
        sub=(3, 3, i),
        min=vmin,
        max=vmax,
        cmap="viridis",
        title=f'{run_name}\nE(B-V) < {EXTINCTION_CUT[run_name]:.3f}: A_eff = {df.loc[run_name, "A_eff [deg2]"]:.0f} deg2',
        unit="coadded i depth (dust corrected)",
        cbar=True,
    )
fig.suptitle("ExgalM5WithCuts footprint of each run (common color scale)", fontsize=16, y=1.02)
save_fig(fig, "EFFAREA_footprint_maps_allruns")
plt.show()

## Caveats

- This effective area is purely geometric: number of pixels that pass the cuts times the pixel area. It carries no seeing or source-density weighting (that was the purpose of the custom n_eff version, kept in `00_EffectiveSurfaceArea_custom_neff_old.ipynb`).
- `n_filters=6` requires visits in all of ugrizy in a pixel, as in the batch. The footprint is therefore smaller than the weak-lensing footprint of notebook 03, which only uses `gri`/`riz`.
- The batch keeps `lim_ebv = 0.2` for every simulation. With `USE_RUN_CUT = True` the metric cut follows the threshold of each run, so the curve mixes the change of the simulated footprint and the change of the metric cut; with `USE_RUN_CUT = False` the metric cut is fixed and only the simulated footprint changes.
- Only year 10 is computed. The batch also evaluates years 1 to 9 with the year-dependent depth cuts `mag_cuts`; the same loop can be added if the time evolution is needed.
- The summary metric is named `Effective Area (deg)` in the batch, but its unit is deg^2 (pixel area in deg^2).

## References
- `rubin_sim/maf/batches/science_radar_batch.py`, block *Cosmology / Static Science* (`ExgalM5WithCuts`, `CountRatioMetric`)
- `01_compareExgalM5withCuts.ipynb` (same metric with other cuts and the ratio to the baseline)